# Continued Hybrid Training — Warm-Start CNN Fine-Tuning

Takes the best checkpoint from a **`parallel`** (CNN+XGB) run and fine-tunes
the CNN alone in **`nn_only`** mode using a two-phase strategy:

| Phase | Layers trained | LR | Epochs |
|---|---|---|---|
| 1 — Head recalibration | `head` only | `HEAD_LR` | `HEAD_FREEZE_EPOCHS` |
| 2 — Full fine-tune | all (`block1/2/3 + head`) | `FINETUNE_LR` | remaining |

**Why**: the CNN in parallel mode learned under a *combined* logit objective
(w≈0.48).  Its feature detectors are strong but the output bias is miscalibrated
for standalone use.  Phase 1 recalibrates the head; phase 2 fine-tunes with a
reduced LR to preserve the learned features.

## 1. Setup

In [ ]:
import sys, os, socket
import torch
import importlib

# Two levels up: HybridTrainV2/ → NNsTorchV2/ → KIprojV2_Claude/ (project root)
sys.path.insert(0, os.path.dirname(os.path.dirname(os.getcwd())))

import NNsTorchV2
importlib.reload(NNsTorchV2)
import NNsTorchV2.HybridTrainV2 as htv2
importlib.reload(htv2)

from NNsTorchV2.HybridTrainV2 import HybridTrainingManager
from NNsTorchV2.HybridTrainV2.components.hybrid_models import build_hybrid_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

def detect_system():
    h = socket.gethostname()
    if h.startswith('VDI0147'): return 'Thermo10'
    if h.startswith('NB'):      return 'Windows'
    return 'GPU'
SYSTEM = detect_system()
print(f'System: {SYSTEM}')

## 2. Configuration

In [ ]:
# ── Mode ─────────────────────────────────────────────────────────────────────
MODE      = 'nn_only'
N_FILTERS = 32

# ── Data ─────────────────────────────────────────────────────────────────────
POWER_MODE  = '4kw_both'
SUBFOLDER   = 'Taris/Data_ML_V3'
MASK_TYPE   = 'alternative'
INVERT_MASK = True
DATA_REGIME = 'postprocessed'
PPT_PHASES  = 'all'
PPT_AMPS    = 6
DIRS        = [0, 1, 2, 3, 4, 5, 6]

# ── Patch / image size ───────────────────────────────────────────────────────
PATCH_MODE  = 'full_padding'
PATCH_SIZE  = (64, 64)

# ── Training ─────────────────────────────────────────────────────────────────
MODEL_NAME = 'cnn_se'
N_SPLITS   = 3
EPOCHS     = 30
PATIENCE   = 15
BATCH_SIZE = 4
LOSS_NAME  = 'tversky'
ALPHA      = 0.3   # Tversky FP penalty  (low = tolerate FP, push recall)
BETA       = 0.7   # Tversky FN penalty  (high = penalise FN for sparse masks)

# ── Warm-start ───────────────────────────────────────────────────────────────
# Directory containing fold{N}_ep{XX}.pt and fold{N}_best.pt
WARMSTART_CKPT_DIR = 'checkpoints/hybrid_parallel/20260304-101704'

# Per-fold starting checkpoint: int = epoch number, 'best' = fold{N}_best.pt
WARMSTART_EPOCHS   = {1: 3, 2: 'best', 3: 'best'}

# Phase 1: only the 1×1 conv head is trained (recalibrates output scale/bias)
HEAD_FREEZE_EPOCHS = 5     # set 0 to skip phase 1 entirely
HEAD_LR            = 1e-3

# Phase 2: full model fine-tune (lower LR to preserve parallel-trained features)
FINETUNE_LR        = 1e-4

# ── XGBoost ──────────────────────────────────────────────────────────────────
XGB_MODEL_PATH = '../../Trees/Trees_database/XGB_setV3[all]_md30minch30gam0.1_lr0.05_n300_normalized_scale2.joblib'

# ── MLflow ───────────────────────────────────────────────────────────────────
MLFLOW_URI = 'sqlite:////tmp/mlflow_experiments/mlflow.db'
print(f'MLflow UI: mlflow ui --backend-store-uri {MLFLOW_URI} --host 127.0.0.1 --port 5000')

## 3. Resolve Warm-Start Checkpoint Paths

In [ ]:
warmstart_ckpt_paths = {}
for fold, ep in WARMSTART_EPOCHS.items():
    if ep == 'best':
        fname = f'fold{fold}_best.pt'
    else:
        fname = f'fold{fold}_ep{int(ep):02d}.pt'
    full_path = os.path.join(WARMSTART_CKPT_DIR, fname)
    exists = os.path.isfile(full_path)
    warmstart_ckpt_paths[fold] = full_path
    print(f'  Fold {fold}: {fname}  {"OK" if exists else "MISSING"}')

missing = [p for p in warmstart_ckpt_paths.values() if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(f'Missing checkpoints: {missing}')
print('All checkpoints found.')

## 4. Initialise Manager

In [ ]:
manager = HybridTrainingManager(
    model_name     = MODEL_NAME,
    sys            = SYSTEM,
    mode           = MODE,
    xgb_model_path = XGB_MODEL_PATH,
    power_mode     = POWER_MODE,
    subfolder_name = SUBFOLDER,
    patch_size     = PATCH_SIZE,
    patch_mode     = PATCH_MODE,
    initial_lr     = FINETUNE_LR,
    loss_name      = LOSS_NAME,
    alpha          = ALPHA,
    beta           = BETA,
    mask_type      = MASK_TYPE,
    dirs           = DIRS,
    ppt_phases     = PPT_PHASES,
    ppt_amps       = PPT_AMPS,
    invert_mask    = INVERT_MASK,
    data_regime    = DATA_REGIME,
    mlflow_uri     = MLFLOW_URI,
    save_chckpnt   = True,
    init_w         = 0
)

def model_fn():
    return build_hybrid_model(MODE, manager.n_raw_ch, N_FILTERS, model_name=MODEL_NAME)

n_params = sum(p.numel() for p in model_fn().parameters())
print(f'Input shape : {manager.input_shape}')
print(f'Model params: {n_params:,}')
print(f'Phase 1 LR  : {HEAD_LR:.1e}  ({HEAD_FREEZE_EPOCHS} epochs, head only)')
print(f'Phase 2 LR  : {FINETUNE_LR:.1e}  ({EPOCHS - HEAD_FREEZE_EPOCHS} epochs, all layers)')

## 5. Warm-Start Fine-Tuning

In [ ]:
fold_metrics, avg = manager.run_kfold(
    model_fn,
    n_splits             = N_SPLITS,
    batch_size           = BATCH_SIZE,
    epochs               = EPOCHS,
    patience             = PATIENCE,
    warmstart_ckpt_paths = warmstart_ckpt_paths,
    head_freeze_epochs   = HEAD_FREEZE_EPOCHS,
    head_lr              = HEAD_LR,
)

## 6. Results

In [ ]:
names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'IoU']
ws_summary = ', '.join(f'fold{k}=ep{v:02d}' if isinstance(v, int) else f'fold{k}={v}'
                       for k, v in WARMSTART_EPOCHS.items())
print(f'Mode: {MODE}  |  Model: {manager.versioned_name}')
print(f'Warm-start: {ws_summary}')
print(f'Phase1: head-only {HEAD_FREEZE_EPOCHS} ep @ {HEAD_LR:.1e}  '
      f'Phase2: full {EPOCHS-HEAD_FREEZE_EPOCHS} ep @ {FINETUNE_LR:.1e}')
print('-' * 60)
for i, fm in enumerate(fold_metrics):
    print(f'Fold {i+1}: ' + '  '.join(f'{n}={v:.4f}' for n, v in zip(names, fm)))
print('=' * 60)
print('Average:')
for n, v in zip(names, avg):
    print(f'  {n:12s}: {v:.4f}')
print(f'\nCheckpoints: {manager.ckpt_dir}')